# Decision Tree / Random Forest Model Tree



## Import Data, libraries, define constants, etc.

In [4]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
import numpy as np
from datetime import datetime

import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder
import statistics as stats
import seaborn as sns

url = "/content/drive/MyDrive/Colab_Notebooks/311Project/BOW Tuning random forest/cleaned_data_bag.csv"
df = pd.read_csv(url)

# separate features for training from validation
df_valid = df[df['is_train'] == False]
df = df[df['is_train'] == True]

# features : array of features in df
features = df.columns.tolist()
features.remove("unique_id")
features.remove('painting')
features.remove('is_train')

PAINTINGS = ["The Persistence of Memory", "The Starry Night", "The Water Lily Pond"]
NUM_TREES = 100


## Function for decision tree

In [6]:
#define function to make tree, called decision tree, from a dataframe X
def decision_tree(df=df, criterion="entropy", max_depth=3, min_samples_leaf=1, max_features="sqrt"):
  """
  run decision tree model on dataframe X
  criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, max_features=max_features
  return the decision tree model that is fit as per the parameters in the function.
  """


  X=df[features]
  t=df["painting"]
  tree = DecisionTreeClassifier(criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, max_features=max_features)
  tree.fit(X, t)
  return tree

## Function for random forest



In [7]:
def construct_forest(df=df, criterion="entropy", max_depth=3, min_samples_leaf=1, max_features="sqrt", boot_size=100):
  """
  construct random forest, <forest>, in the form of a list of decision trees.
  fit each tree on a unique bootstrapped sample of <df> with replacement and size <boot_size>.
  return <forest>.
  """
  forest = []
  for t in range(NUM_TREES):
    np.random.seed(311)
    df_boot = df.sample(n=boot_size, replace=True)
    forest.append(decision_tree(df=df_boot, criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, max_features=max_features))
  return forest

## Hypeparameter training


In [ ]:
# Save to the CSV

hyperparameters = {
    "criterion": [0, 2],
    "max_depth": [1, 16],
    "min_samples_leaf": [1, 15],
    "max_features": [0, 4],
    "boot_size": [len(df)/2, len(df)]
    }
CRITERION = ["gini", "entropy"]
NUM_FEATURES = ["sqrt", "log2", 0.33, 1]

tuning = pd.DataFrame(columns=["Criterion", "Max Depth", "Min Sample Leaf", "Max Features", "Boot Size", "T Acc", "V Acc", "V Pred", "V Rec", "V F1"])
for i in range(1000):
  #for any param_config
  np.random.seed(int(datetime.now().timestamp()))
  c = CRITERION[np.random.randint(hyperparameters["criterion"][0], hyperparameters["criterion"][1])]
  md = np.random.randint(hyperparameters["max_depth"][0], hyperparameters["max_depth"][1])
  msl = np.random.randint(hyperparameters["min_samples_leaf"][0], hyperparameters["min_samples_leaf"][1])
  mf = NUM_FEATURES[np.random.randint(hyperparameters["max_features"][0], hyperparameters["max_features"][1])]
  bs = np.random.randint(hyperparameters["boot_size"][0], hyperparameters["boot_size"][1])
  le = LabelEncoder()
  df["painting"] = le.fit_transform(df["painting"])
  df_valid["painting"] = le.transform(df_valid["painting"])

  np.random.seed(311)
  forest = construct_forest(df=df, criterion=c, max_depth=md, min_samples_leaf=msl, max_features=mf, boot_size=bs)

  votes = np.stack([tree.predict(df[features]) for tree in forest])
  preds = np.apply_along_axis(stats.mode, 0, votes)

  v_votes = np.stack([tree.predict(df_valid[features]) for tree in forest])
  v_preds = np.apply_along_axis(stats.mode, 0, v_votes)

  t_acc = metrics.accuracy_score(df['painting'].to_numpy(), preds)
  v_acc = metrics.accuracy_score(df_valid["painting"].to_numpy(), v_preds)
  v_prec = metrics.precision_score(df_valid["painting"].to_numpy(), v_preds, average="macro")
  v_rec = metrics.recall_score(df_valid["painting"].to_numpy(), v_preds, average="macro")
  v_f1 = metrics.f1_score(df_valid["painting"].to_numpy(), v_preds, average="macro")

  result = pd.DataFrame([{"Criterion" : c, "Max Depth": md, "Min Sample Leaf": msl, "Max Features": mf, "Boot Size": bs, "T Acc": t_acc, "V Acc": v_acc, "V Pred": v_prec, "V Rec": v_rec, "V F1": v_f1}])
  tuning = pd.concat([tuning, result])

pd.DataFrame(tuning).to_csv("results.csv")



/tmp/ipykernel_12418/3516782273.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tuning = pd.concat([tuning, result])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(res

## Brenden fooling around

In [ ]:
plt.scatter(tuning['Boot Size'], tuning['V Acc'])
plt.show()

In [ ]:
plt.scatter(tuning['Max Depth'], tuning['V Acc'])
plt.show()

In [ ]:
plt.scatter(tuning['Min Sample Leaf'], tuning['V Acc'])
plt.show()


In [ ]:
sns.boxplot(data=tuning, x="V Acc", y="Criterion" )
plt.show()

In [ ]:
sns.boxplot(data=tuning, x="V Acc", hue="Max Features" )
plt.show()

In [ ]:
forest = construct_forest(max_depth=3)
votes = []
for tree in forest:
  votes.append(tree.predict(df[features])[120])
print(votes)
print(stats.mode(votes))

In [ ]:
from sklearn.tree import export_text
text_representation = export_text(forest[2], feature_names=features)
print(text_representation)
tree = forest[5]
left_children = tree.tree_.children_left
right_children = tree.tree_.children_right
feature_splits = tree.tree_.feature # array of the feature index used for splitting
thresholds = tree.tree_.threshold # array of the split points/thresholds

In [ ]:
feature_splits

In [ ]:
thresholds

In [ ]:
left_children

In [ ]:

url = "/content/drive/MyDrive/Colab_Notebooks/311Project/BOW Tuning random forest/BOWresults0-1500 - BOWresults300.csv"
df_bow = pd.read_csv(url)

In [ ]:
plt.scatter(df_bow['Boot Size'], df_bow['V Acc'])
plt.show()

In [ ]:
sns.violinplot(data=df_bow, x=df_bow['Max Depth'], y=df_bow['V Acc'])
plt.show()

In [ ]:
sns.violinplot(data=df_bow, x=df_bow['Min Sample Leaf'], y=df_bow['V Acc'])
plt.show()

In [ ]:
sns.boxplot(data=df_bow, x=df_bow['Max Features'], y=df_bow['V Acc'])
plt.show()

In [ ]:
sns.violinplot(data=df_bow, x=df_bow['Criterion'], y=df_bow['V Acc'])
plt.show()